# __Azure Connection and Data Retrieval__

> This notebook is creaeted as a experiment for learning the conection of python and Azure Data Lake Storage Gen2 (ADLS Gen2) and allow acces to cloud files.

## __1. Requirements__

1. Python Virtual Environment  
2. Azure Dependences (downloaded)
3. Azure Account configured with credentials-based authentication

## __2. Autentication__

In [2]:
import pandas as pd
# Access an ADLS filesystem
from azure.identity import DefaultAzureCredential

# Autentication Object
credential = DefaultAzureCredential()

In [4]:
# Verify Azure authentication
token = credential.get_token("https://storage.azure.com/.default")

print("Authentication successful")
print(f"Token expires at: {token.expires_on}")

Authentication successful
Token expires at: 1791947570


## __3. Storage Account client__

In [6]:

# Import the ADL client
from azure.storage.filedatalake import DataLakeServiceClient

# module to access environment variables
import os 

# Get the Storage Account name from the environment variabels 
account_name = os.environ['AZURE_STORAGE_ACCOUNT']
# Build the URL of the ADLS account
account_url = f"https://{account_name}.dfs.core.windows.net"

# Create the Dta Lake services client (using the account URL and credentials)
client = DataLakeServiceClient(
    account_url= account_url, 
    credential=credential
)

In [ ]:
# Storage Account URL to verify the configuration
print(account_url)

# Client type to verify that it was created
print(type(client).__name__)

https://datafinanceproject.dfs.core.windows.net
DataLakeServiceClient


## __4. Access to filesystem__

In [9]:
# Get a client for 'interim' filesystem
filesystem = client.get_file_system_client('interim')

# verify the connection 
filesystem.file_system_name

'interim'

In [ ]:
# Retrieve the paths of the items in the filesystem
paths = filesystem.get_paths()

# files on 'Interim'
for path in paths:
    print(path.name)

financial_statements
financial_statements/2021
financial_statements/2021/balances_2021_1.txt
financial_statements/2021/balances_2021_2.txt
financial_statements/2021/balances_2021_3.txt
financial_statements/2021/catalogo_2021_1.txt
financial_statements/2021/catalogo_2021_2.txt
financial_statements/2021/catalogo_2021_3.txt
financial_statements/2022
financial_statements/2022/balances_2022_1.txt
financial_statements/2022/balances_2022_2.txt
financial_statements/2022/balances_2022_3.txt
financial_statements/2022/catalogo_2022_1.txt
financial_statements/2022/catalogo_2022_2.txt
financial_statements/2022/catalogo_2022_3.txt
financial_statements/2023
financial_statements/2023/balances_2023_1.txt
financial_statements/2023/balances_2023_2.txt
financial_statements/2023/catalogo_2023_1.txt
financial_statements/2023/catalogo_2023_2.txt
financial_statements/2024
financial_statements/2024/balances_2024_1.txt
financial_statements/2024/catalogo_2024_1.txt
financial_statements/2025
financial_statements/

## __5. File Reading__

In [11]:
# Client for the specific file
file_client = filesystem.get_file_client(
    "financial_statements/2025/balances_2025_1.txt"
)

type(file_client)

azure.storage.filedatalake._data_lake_file_client.DataLakeFileClient

In [ ]:
# Get the file
download = file_client.download_file()

In [13]:
# Read (1000 first bytes)
download.read(1000)

b'A\xd1O\tEXPEDIENTE\tRUC\tNOMBRE\tRAMA_ACTIVIDAD\tDESCRIPCION_RAMA\tCIIU\tCUENTA_1\tCUENTA_101\tCUENTA_10101\tCUENTA_1010101\tCUENTA_1010102\tCUENTA_1010103\tCUENTA_10102\tCUENTA_1010201\tCUENTA_101020101\tCUENTA_10102010101\tCUENTA_10102010102\tCUENTA_10102010103\tCUENTA_10102010104\tCUENTA_10102010105\tCUENTA_10102010106\tCUENTA_101020102\tCUENTA_10102010201\tCUENTA_10102010202\tCUENTA_10102010203\tCUENTA_10102010204\tCUENTA_10102010205\tCUENTA_10102010206\tCUENTA_10102010207\tCUENTA_10102010208\tCUENTA_10102010209\tCUENTA_10102010210\tCUENTA_10102010211\tCUENTA_10102010212\tCUENTA_10102010213\tCUENTA_10102010214\tCUENTA_10102010215\tCUENTA_10102010216\tCUENTA_10102010217\tCUENTA_10102010218\tCUENTA_10102010219\tCUENTA_10102010220\tCUENTA_10102010221\tCUENTA_10102010222\tCUENTA_10102010223\tCUENTA_101020103\tCUENTA_10102010301\tCUENTA_10102010302\tCUENTA_10102010303\tCUENTA_10102010304\tCUENTA_1010202\tCUENTA_101020201\tCUENTA_10102020101\tCUENTA_10102020102\tCUENTA_10102020103\tCUE

In [15]:
# To treat text as a file-like object in memory
from io import StringIO
import pandas as pd

download = file_client.download_file()
# Read downloaded bytes and decode them using latin-1
content = download.readall().decode("latin1")

# Read Data
data1 = pd.read_csv(
    StringIO(content),
    sep="\t",
    decimal=",",
    low_memory=False
)

In [19]:
data1.head(4)

,AÑO,EXPEDIENTE,RUC,NOMBRE,RAMA_ACTIVIDAD,DESCRIPCION_RAMA,CIIU,CUENTA_1,CUENTA_101,CUENTA_10101,...,CUENTA_80004,CUENTA_80005,CUENTA_80006,CUENTA_80007,CUENTA_80008,CUENTA_80009,CUENTA_801,CUENTA_80101,CUENTA_80102,Unnamed: 629
0,2025,1,1790013731001,ACEITES TROPICALES SOCIEDAD ANONIMA ATSA,A,"AGRICULTURA, GANADERÍA, SILVICULTURA Y PESCA.",A0126.01,1.329499e+06,9.638320e+03,6338.46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,2025,2,1790004724001,ACERIA DEL ECUADOR CA ADELCA.,C,INDUSTRIAS MANUFACTURERAS.,C2410.25,5.264843e+08,2.481542e+08,28574349.24,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,2025,3,1790008959001,ACERO COMERCIAL ECUATORIANO S.A.,G,COMERCIO AL POR MAYOR Y AL POR MENOR REPARACIÓ...,G4610.03,1.206475e+07,1.180430e+07,1505408.28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,2025,11,1790044149001,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,H,TRANSPORTE Y ALMACENAMIENTO.,H5110.01,1.718426e+07,1.717797e+07,191024.71,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


---